In [1]:

import os

GROQ_API_KEY = input("Enter your Groq API key: ").strip()
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print(" API key set successfully!")

✅ API key set successfully!


In [2]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import CharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.callbacks import StdOutCallbackHandler
import os
import textwrap

In [5]:
pdf_path = "./Introduction_to_Data_and_Data_Science.pdf"
docx_path = "./Introduction_to_Data_and_Data_Science_2.docx"

# PDF loader
pdf_loader = PyPDFLoader(pdf_path)
pdf_docs = pdf_loader.load()
print("Loaded PDF pages:", len(pdf_docs))

# DOCX loader
docx_loader = Docx2txtLoader(docx_path)
docx_docs = docx_loader.load()
print("Loaded DOCX docs:", len(docx_docs))

Loaded PDF pages: 6
Loaded DOCX docs: 1


In [6]:
char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=3000,
    chunk_overlap=200,
    length_function=len
)

pdf_chunks = char_splitter.split_documents(pdf_docs)
docx_chunks = char_splitter.split_documents(docx_docs)

print("PDF chunks:", len(pdf_chunks), "DOCX chunks:", len(docx_chunks))

Created a chunk of size 3239, which is longer than the specified 3000


PDF chunks: 6 DOCX chunks: 4


In [8]:
emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Test a sample query embedding
sample_vec = emb.embed_query("hello world")
print("✅ Sample embedding length:", len(sample_vec))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\JayKalsariya\PycharmProjects\08_The_AI_Engineer_Course\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JayKalsariya\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Sample embedding length: 384


In [9]:
persist_dir = "./rag_chroma_db"
os.makedirs(persist_dir, exist_ok=True)

# Combine all document chunks
all_chunks = pdf_chunks + docx_chunks

# Initialize Chroma vector store
vectordb = Chroma.from_documents(
    documents=all_chunks,
    embedding=emb,
    persist_directory=persist_dir
)

# For Chroma 0.5+, persist is automatic
print("Chroma vectorstore persisted at:", persist_dir)

# Verify stored vector count
try:
    print("Stored vectors count:", vectordb._collection.count())
except Exception:
    print("⚠️ Count unavailable in current Chroma version")


Chroma vectorstore persisted at: ./rag_chroma_db
Stored vectors count: 10


In [10]:
vectordb = Chroma(
    persist_directory=persist_dir,
    embedding_function=emb
)
print("Reloaded Chroma DB from:", persist_dir)


query = "Summarize the main conclusions of the document."
print(f"\n Query: {query}\n")

try:
    results = vectordb.similarity_search(query, k=3)
    for i, doc in enumerate(results, 1):
        print(f"--- Result {i} ---")
        print(doc.page_content[:500].strip(), "...\n")
        print("metadata:", doc.metadata)
        print()
except Exception as e:
    print("⚠ Similarity search failed:", e)


retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print(" Retriever object ready for RAG chain construction.")

# Check vector count
try:
    print("Stored vectors:", vectordb._collection.count())
except Exception:
    print("Vector count not available.")


Reloaded Chroma DB from: ./rag_chroma_db

 Query: Summarize the main conclusions of the document.

--- Result 1 ---
We hope we gave you a good idea about the 
level of applicability of the most frequently 
used programming and software tools in the 
field of data science. 
Thank you for watching! ...

metadata: {'creator': 'Microsoft® Word for Microsoft 365', 'page_label': '6', 'producer': 'Microsoft® Word for Microsoft 365', 'source': './Introduction_to_Data_and_Data_Science.pdf', 'author': 'Hristina  Hristova', 'page': 5, 'moddate': '2023-11-09T10:16:34+02:00', 'total_pages': 6, 'creationdate': '2023-11-09T10:16:34+02:00'}

--- Result 2 ---
# Introduction to Data and Data Science

## Analysis vs Analytics ...

metadata: {'source': './Introduction_to_Data_and_Data_Science_2.docx'}

--- Result 3 ---
and exploring what you could do with them 
in the future. 
Here, analytics branches off into two areas: 
qualitative analytics – this is using your 
intuition and experience in conjunction 

In [11]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=1024,
    timeout=60,
)

# 🧠 Prompt template
prompt_template = ChatPromptTemplate.from_template("""\
You are a helpful assistant that answers based on the provided context.

Context:
{context}

Question:
{question}

Answer clearly and concisely using only information from the context.
If the context is insufficient, say you don't have enough information.
""")

# 🔗 Build RAG pipeline
rag_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt_template
    | llm
    | StrOutputParser()
)


In [14]:
query = "What are the main findings discussed in the uploaded documents?"
print(" Query:", query)
print("\n RAG Response:\n")
response = rag_chain.invoke(query)
print(response)

 Query: What are the main findings discussed in the uploaded documents?

 RAG Response:

The main findings discussed in the uploaded documents are:

1. The most frequently used programming languages in data science are R and Python, due to their adaptability and ability to manipulate data.
2. SQL is specifically used for relational database management systems and is advantageous when working with traditional, historical data.
3. MATLAB is ideal for working with mathematical functions or matrix manipulations, but its usage is limited due to being a paid service.
4. Other programming languages like Java, Scala, JavaScript, C, and C++ are used in big data and machine learning.
5. Various software tools like Excel, SPSS, Apache Hadoop, Power BI, Tableau, EViews, and Stata are used in different areas of data science, such as traditional data, business intelligence, and predictive analytics.

These findings highlight the importance of programming languages and software tools in the field of 

In [15]:
llm_stream = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=1024,
    streaming=True,
    callbacks=[StdOutCallbackHandler()]
)

rag_stream_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt_template
    | llm_stream
    | StrOutputParser()
)

print("\n Streaming Query:", query)
print("\n Streaming response (Groq):\n")
response = rag_stream_chain.invoke(query)

# --- Retrieved context snippets ---
print("\n Retrieved context snippets:\n")
source_docs = retriever.invoke(query)

for i, doc in enumerate(source_docs, 1):
    snippet = textwrap.shorten(doc.page_content.strip(), width=350, placeholder="...")
    print(f"[{i}] {snippet}")
    if "source" in doc.metadata:
        print("   ↳ metadata:", doc.metadata)
    print()


 Streaming Query: What are the main findings discussed in the uploaded documents?

 Streaming response (Groq):


 Retrieved context snippets:

[1] We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!
   ↳ metadata: {'page_label': '6', 'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2023-11-09T10:16:34+02:00', 'creationdate': '2023-11-09T10:16:34+02:00', 'page': 5, 'source': './Introduction_to_Data_and_Data_Science.pdf', 'author': 'Hristina  Hristova', 'total_pages': 6}

[2] # Introduction to Data and Data Science ## Analysis vs Analytics
   ↳ metadata: {'source': './Introduction_to_Data_and_Data_Science_2.docx'}

[3] ## Programming Languages & Software Employed in Data Science - All the Tools You Need
   ↳ metadata: {'source': './Introduction_to_Data_and_Data_Science_2.docx'}

[4] Alright! So… How are 

In [17]:
prompt_cited = ChatPromptTemplate.from_template("""\
You are a helpful assistant that answers based ONLY on the provided context.

Context (each item labeled numerically):
{context}

Question:
{question}

When using any piece of information, cite its source by number in square brackets, e.g. [1], [2].
Provide a coherent, factual answer grounded strictly in the context.
""")

rag_cited_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt_cited
    | llm
    | StrOutputParser()
)

query = "Summarize all important findings with proper inline citations."
print("\n Query with citations:", query)
response_cited = rag_cited_chain.invoke(query)
print("\n Answer with Citations:\n")
print(response_cited)


 Query with citations: Summarize all important findings with proper inline citations.

 Answer with Citations:

In the field of data science, programming languages and software tools play a crucial role [4]. The most popular programming languages used are R and Python, which can manipulate data and are integrated within multiple data and data science software platforms [4]. They are adaptable and can solve a wide variety of business and data-related problems [4]. However, they have limitations, and other languages like SQL are more suitable for specific domains, such as relational database management systems [4]. 

SQL is specifically created for working with traditional, historical data and is often employed in business intelligence (BI) analysis [4]. MATLAB is another language that is ideal for working with mathematical functions or matrix manipulations, but its usage is a paid service, and it is losing ground to open-source languages like R and Python [4]. 

In the area of big data

In [19]:
class RAGPipeline:
    """Reusable end-to-end Retrieval-Augmented Generation workflow with Groq."""

    def __init__(self, retriever, model_name="llama-3.3-70b-versatile", temperature=0.3):
        self.retriever = retriever
        self.llm = ChatGroq(
            model=model_name,
            temperature=temperature,
            max_tokens=1024,
            timeout=60,
        )
        self.prompt = ChatPromptTemplate.from_template("""\
        You are a helpful assistant. Use only the context to answer.

        Context:
        {context}

        Question:
        {question}

        Provide a clear, factual answer. If context lacks data, say so.
        """)
        self.chain = (
            RunnableParallel({"context": self.retriever, "question": RunnablePassthrough()})
            | self.prompt
            | self.llm
            | StrOutputParser()
        )

    def query(self, question: str) -> str:
        return self.chain.invoke(question)

# Instantiate and test
rag_pipeline = RAGPipeline(retriever)
answer = rag_pipeline.query("What are the overall insights across the uploaded documents?")
print("\n Final Answer from RAG Pipeline:\n", answer)



 Final Answer from RAG Pipeline:
 The overall insights across the uploaded documents are related to the field of data science, specifically the distinction between "analysis" and "analytics". 

Analysis refers to the process of breaking down a dataset into smaller parts to study and understand what has happened in the past. It is used to explain how and why something occurred.

Analytics, on the other hand, refers to the application of logical and computational reasoning to explore potential future events. It branches off into two areas: qualitative analytics, which uses intuition and experience to plan business moves, and quantitative analytics, which applies formulas and algorithms to numbers gathered from analysis.

The documents also touch on the idea that terms such as "data analysis", "data analytics", "business analysis", and "business analytics" have unique meanings, but a more detailed explanation of these terms is promised in a subsequent video. 

No specific data or numeric

In [20]:
print("\n" + "="*50)
print(" Interactive RAG Q&A Session")
print("="*50)
print("Type 'quit' to exit\n")

while True:
    question = input("\n❓ Your question: ").strip()
    if question.lower() == 'quit':
        break

    print("\n Thinking...")
    answer = rag_pipeline.query(question)
    print(f"\n Answer: {answer}\n")
    print("-"*50)


 Interactive RAG Q&A Session
Type 'quit' to exit


 Thinking...

 Answer: This document is about introducing the programming languages and software tools used in the field of data science, including their applicability, advantages, and limitations. It covers various tools such as R, Python, MATLAB, SQL, Java, Scala, Excel, SPSS, Hadoop, Power BI, and others, and explains how they are used in different areas of data science, including traditional data, business intelligence, big data, and predictive analytics. The document aims to provide an overview of the most frequently used tools in data science and their level of applicability.

--------------------------------------------------

 Thinking...

 Answer: The context provided does not contain information about metricas multiplication or how to reduce its computational complexity to sqrt(2 m n k) FLOPS. The context primarily discusses various programming languages and software tools used in data science, such as R, Python, MATLAB, and